# d2_submit — Direction 2 submission: MPC on a building model rebuilt from the metadata

Internet **off**. Inputs: the competition data and the dataset made from `python -m hvac_controller.d2.package`
(a zip holding `hvac_controller/`). The simulator the model is rebuilt with (`smart_buildings`) comes from the
competition data. Run interactively, it plays the public example (`building_train_1/atlanta_2`); submitted, the same
notebook is re-run on the hidden scenarios.

In [ ]:
import sys, time
from pathlib import Path

INPUT = Path("/kaggle/input")
candidates = [INPUT / "competitions" / "neurips-2026-smart-buildings", INPUT / "neurips-2026-smart-buildings"]
candidates += [p.parent for p in sorted(INPUT.rglob("kaggle_evaluation")) if (p.parent / "smart_buildings").exists()]
DATA_ROOT = next(p for p in candidates if (p / "smart_buildings").exists() and (p / "kaggle_evaluation").exists())
CODE = next(p.parents[2] for p in sorted(INPUT.rglob("hvac_controller/d2/submission.py")))
sys.path.insert(0, str(DATA_ROOT))
sys.path.insert(0, str(DATA_ROOT / "kaggle_evaluation"))
sys.path.insert(0, str(CODE))
print("competition data:", DATA_ROOT, "| code:", CODE, "|", (CODE / "README.txt").read_text() if (CODE / "README.txt").exists() else "")

import torch
torch.set_num_threads(1)
from hvac_controller.d2.submission import D2Policy
from kaggle_evaluation import smart_buildings_inference_server

T0 = time.time()


class SubmissionPolicy(smart_buildings_inference_server.Policy):
    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        print(f"[d2] scenario start at {(time.time() - T0) / 60:.1f} min", flush=True)
        self.policy = D2Policy(building_meta_data, action_spec)

    def get_action(self, obs):
        return self.policy.get_action(obs)

In [ ]:
smart_buildings_inference_server.SmartBuildingsInferenceServer(SubmissionPolicy).run(
    competition_data_folder=str(DATA_ROOT),
)
print(f"[d2] done after {(time.time() - T0) / 60:.1f} min")